# Bonus (optionnel, avancé) — Autoencodeurs parcimonieux & signal émergent

Cette piste n'est **pas obligatoire** pour le livrable principal — elle est là si votre
groupe termine en avance et veut voir ce que l'apprentissage de représentation non
supervisé peut découvrir, avec **zéro** étiquette, à l'intérieur d'un modèle de langage
génomique.

## Partie 1 — Rappel : l'autoencodeur classique

Un autoencodeur apprend à compresser son entrée à travers un **goulot d'étranglement**
(bottleneck) étroit et à la reconstruire — ce goulot le force à ne garder que la structure
la plus utile. Voyons cela d'abord sur nos propres vecteurs de k-mers, car c'est rapide et
ne nécessite rien d'Evo2.

In [ ]:
import sys
sys.path.append("src")

import torch
import torch.nn as nn
from data import load_all
from featurize import kmer_matrix

train_df = load_all("../../../data/supervised/processed")["train"]
X = torch.tensor(kmer_matrix(train_df["sequence"].tolist(), k=4), dtype=torch.float32)

d_in = X.shape[1]
vanilla_ae = nn.Sequential(
    nn.Linear(d_in, 16), nn.ReLU(),          # <- goulot : 256 kmers -> 16 nombres
    nn.Linear(16, d_in),
)
optimizer = torch.optim.Adam(vanilla_ae.parameters(), lr=1e-3)

for epoch in range(20):
    optimizer.zero_grad()
    x_hat = vanilla_ae(X)
    loss = nn.functional.mse_loss(x_hat, X)
    loss.backward()
    optimizer.step()
print(f"final reconstruction loss: {loss.item():.5f}")

Ce goulot d'étranglement est **dense** : chacun des 16 nombres est utilisé
pour chaque entrée, et ils ne sont pas individuellement interprétables — une seule
« caractéristique » est généralement un mélange de nombreuses causes sous-jacentes (c'est
la *polysémanticité*).

## Partie 2 — Ce qu'apporte un autoencodeur parcimonieux

Un autoencodeur **parcimonieux** (SAE) fait l'inverse : au lieu d'un goulot *étroit*, il
utilise un goulot **surcomplet** (plus d'unités cachées que d'entrées), mais force
seulement une poignée (`k`) d'entre elles à être actives pour une entrée donnée. L'idée
(issue de recherches récentes en interprétabilité mécanistique sur les modèles de
langage) : avec suffisamment de capacité et la bonne parcimonie, les caractéristiques
individuelles ont tendance à devenir **monosémantiques** — chacune suit un motif
distinct, ce qui les rend inspectables.

Ici, nous entraînons un SAE sur des **activations brutes d'Evo2** — des états internes
par position de nucléotide, extraits sans aucune information d'étiquette — et nous
cherchons des caractéristiques qui correspondent à quelque chose de réel, comme le cadre
de lecture d'un codon (structure de période 3).

In [ ]:
from embeddings import load_autoencoder_activations
from models.sae import TopKSparseAutoencoder, train_sae

acts = load_autoencoder_activations("../../../data/autoencoder", "train")
print("full activation dump shape:", acts.shape)

N_SUBSAMPLE = 20_000
X_acts = torch.tensor(acts[:N_SUBSAMPLE].astype("float32"))
print("training on:", X_acts.shape)

In [ ]:
sae = TopKSparseAutoencoder(d_in=X_acts.shape[1], d_hidden=X_acts.shape[1] * 8, k=32)
sae, history = train_sae(sae, X_acts, epochs=10, batch_size=512)

## Partie 3 — Cherchons une structure émergente : la périodicité

L'ADN est lu trois nucléotides à la fois (codons). Si le SAE a découvert quelque chose sur
le cadre de lecture sans aucune supervision, certaines caractéristiques devraient
s'activer périodiquement avec une période ~3 le long d'un segment continu de positions.

In [ ]:
import numpy as np
from viz import plot_sae_feature_activity

window = torch.tensor(acts[1000:1000 + 300].astype("float32"))
with torch.no_grad():
    _, features = sae(window)
features = features.numpy()

variances = features.var(axis=0)
top_features = np.argsort(variances)[::-1][:5]

plot_sae_feature_activity(features[:, top_features], top_features)

In [ ]:
for feat_idx in top_features:
    signal = features[:, feat_idx] - features[:, feat_idx].mean()
    spectrum = np.abs(np.fft.rfft(signal))
    freqs = np.fft.rfftfreq(len(signal))
    period = 1 / freqs[np.argmax(spectrum[1:]) + 1] if len(spectrum) > 1 else float("nan")
    print(f"feature {feat_idx}: dominant period ~ {period:.2f} positions")

### Discussion

- Une caractéristique est-elle ressortie avec une période proche de 3 ? Cela suggérerait
  que le modèle — sans étiquette, sans tâche, juste « reconstruire mes propres
  activations internes » — a représenté le cadre de lecture en interne.
- Essayez de corréler les principales caractéristiques avec les étiquettes
  codant/non-codant de `data/supervised/processed` pour la même région du génome (il
  faudra aligner les positions de tokens avec les coordonnées du génome) — certaines
  caractéristiques séparent-elles les régions codantes des non-codantes toutes seules ?
- C'est exploratoire, non noté sur l'obtention d'un « vrai » résultat — l'important est de
  voir qu'une structure interprétable *peut* émerger d'un entraînement purement non
  supervisé.